# **Agente con LangChain + OpenRouter que crea y consulta su propio RAG desde Wikipedia**



El agente puede traer info de Wikipedia y

*   El agente puede traer info de Wikipedia y guardarla como documento local


*   Mantiene una “mini base de conocimiento” en /content/docs
*   Hace RAG (retrieval + respuesta con fuentes)


*   Puede añadir notas y generar un quiz

*   Puede comparar dos animales












# Instalación

In [ ]:
!pip -q install "langchain==0.2.16" "langchain-core==0.2.38" "langchain-community==0.2.16" "langchain-openai==0.1.23" requests

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-groq 1.1.2 requires langchain-core<2.0.0,>=1.2.8, but you have langchain-core 0.2.38 which is incompatible.
langchain-classic 1.0.1 requires langchain-core<2.0.0,>=1.2.5, but you have langchain-core 0.2.38 which is incompatible.
langchain-classic 1.0.1 requires langchain-text-splitters<2.0.0,>=1.1.0, but you have langchain-text-splitters 0.2.4 which is incompatible.
langgraph-prebuilt 1.0.8 requires langchain-core>=1.0.0, but you have langchain-core 0.2.38 which is incompatible.


In [ ]:
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate

# Configurar OpenRouter

In [ ]:
import os
from langchain_openai import ChatOpenAI

os.environ["OPENAI_API_KEY"] = "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"

llm = ChatOpenAI(
    model="openrouter/free",
    base_url="https://openrouter.ai/api/v1",
    temperature=0,
    default_headers={
        "HTTP-Referer": "https://colab.research.google.com",
        "X-OpenRouter-Title": "Demo Agents + Wikipedia RAG",
    },
)

resp = llm.invoke("Responde en una frase: ¿qué es un agente en IA?")
print(resp.content)

Un agente en IA es un sistema autónomo que ejecuta tareas o toma decisiones sin intervención humana directa.


# Crear la carpeta de conocimiento local

Aquí guardaremos los documentos que el agente vaya creando.

In [ ]:
from pathlib import Path

BASE_PATH = Path("/content/docs")
BASE_PATH.mkdir(parents=True, exist_ok=True)

BASE_PATH

PosixPath('/content/docs')

# Tools: ingesta (Wikipedia), edición (notas) y RAG (search/answer)

Tools funciones:



*   fetch_wikipedia(animal): descarga un resumen y lo guarda como .md

*  ensure_animal(animal): si no existe doc local, lo crea desde Wikipedia

*   rag_search(query): retrieval (devuelve snippets + archivos)
*   rag_answer(question): retrieval + generación (responde con fuentes)


*   add_note(animal, note): añade conocimiento manual al doc


*   make_quiz(animal, n): genera preguntas tipo test basadas SOLO en el doc

* compare_animals(a, b): comparación con evidencia

In [ ]:
import re
import requests
from datetime import datetime
from langchain_core.tools import tool

def _safe_name(s: str) -> str:
    s = s.strip().lower()
    s = re.sub(r"[^a-z0-9ñáéíóúü\- _]", "", s)
    s = re.sub(r"\s+", "_", s)
    return s[:60] if s else "unknown"

def _find_doc_for(animal: str, lang: str = "es"):
    # Busca un doc que empiece por animal_ y contenga el nombre "safe"
    key = _safe_name(animal)
    candidates = sorted(BASE_PATH.glob(f"animal_*_{lang}.md"))
    for fp in candidates:
        if key in fp.name:
            return fp
    return None

from langchain_core.tools import tool
import requests
from datetime import datetime

WIKI_HEADERS = {
    "User-Agent": "DemoLangChainAgents/1.0 (educacion; contact: demo@example.com)",
    "Accept": "application/json",
}

@tool
def fetch_wikipedia(animal: str, lang: str = "es") -> str:
    """
    Descarga el resumen de Wikipedia y lo guarda como Markdown.
    Añade headers (User-Agent) para evitar 403.
    """
    title = (animal or "").strip()
    if not title:
        return "[ERROR] animal vacío"

    url = f"https://{lang}.wikipedia.org/api/rest_v1/page/summary/{requests.utils.quote(title)}"
    r = requests.get(url, headers=WIKI_HEADERS, timeout=20)

    if r.status_code != 200:
        return f"[ERROR] Wikipedia status={r.status_code} para '{title}' (lang={lang})."

    data = r.json()
    page_title = data.get("title", title)
    extract = data.get("extract", "")
    page_url = (data.get("content_urls", {}) or {}).get("desktop", {}).get("page", "")

    filename = f"animal_{_safe_name(page_title)}_{lang}.md"
    fp = BASE_PATH / filename

    md = []
    md.append(f"# {page_title}")
    md.append("")
    md.append(f"Fuente: {page_url}" if page_url else "Fuente: (sin URL)")
    md.append("")
    md.append("## Resumen (Wikipedia)")
    md.append(extract if extract else "(sin extract)")
    md.append("")
    md.append("## Metadata")
    md.append(f"- lang: {lang}")
    md.append(f"- fetched_at: {datetime.utcnow().isoformat()}Z")
    md.append("")

    fp.write_text("\n".join(md), encoding="utf-8")
    return f"OK: guardado {fp.name}"


@tool
def ensure_animal(animal: str, lang: str = "es") -> str:
    """
    Si existe doc local del animal, no hace nada.
    Si no existe, lo crea llamando a fetch_wikipedia.
    Blindado: si lang no es str, fuerza 'es'.
    """
    if not isinstance(lang, str):
        lang = "es"

    fp = _find_doc_for(animal, lang=lang)
    if fp and fp.exists():
        return f"OK: ya existe {fp.name}"
    return fetch_wikipedia.invoke({"animal": animal, "lang": lang})

@tool
def add_note(animal: str, note: str, lang: str = "es") -> str:
    """Añade una nota manual al documento local del animal (crea uno mínimo si no existe)."""
    if not (animal or "").strip():
        return "[ERROR] animal vacío"
    if not (note or "").strip():
        return "[ERROR] note vacía"

    fp = _find_doc_for(animal, lang=lang)
    if fp is None:
        # Creamos doc mínimo si no existía
        filename = f"animal_{_safe_name(animal)}_{lang}.md"
        fp = BASE_PATH / filename
        fp.write_text(f"# {animal}\n\nFuente: (manual)\n\n## Notas\n", encoding="utf-8")

    stamp = datetime.utcnow().isoformat() + "Z"
    with fp.open("a", encoding="utf-8") as f:
        f.write("\n")
        f.write("## Nota añadida\n")
        f.write(f"- added_at: {stamp}\n")
        f.write(note.strip() + "\n")

    return f"OK: nota añadida en {fp.name}"

# RAG “lite”: retrieval (snippets) y respuesta con fuentes



*   rag_search = retrieval (no “redacta”, solo recupera evidencia)
*   rag_answer = retrieval + generación (responde usando SOLO evidencia)






In [ ]:
import re
from langchain_core.tools import tool

STOPWORDS_ES = {
    "el","la","los","las","un","una","unos","unas","de","del","al","y","o","u",
    "que","qué","como","cómo","cual","cuál","cuáles","cuanta","cuántas","cuánto","cuántos",
    "es","son","ser","estar","hay","haber","en","a","por","para","con","sin","sobre",
    "hoy","día","días","actualmente","ahora","segun","según","documentos","locales"
}

def _tokenize_es(text: str) -> list[str]:
    text = (text or "").lower()
    text = re.sub(r"[^a-z0-9ñáéíóúü\s]", " ", text)
    toks = [t for t in text.split() if len(t) >= 3 and t not in STOPWORDS_ES]
    return toks

@tool
def rag_search(query: str, k: int = 4) -> str:
    """
    Retrieval robusto sin embeddings:
    - tokeniza la query
    - score = suma de ocurrencias de tokens en cada doc
    - devuelve snippets del doc con mejor score
    """
    q = (query or "").strip()
    if not q:
        return "[ERROR] query vacía"

    q_tokens = _tokenize_es(q)
    if not q_tokens:
        return "[ERROR] query sin tokens útiles (demasiadas stopwords)"

    hits = []
    for fp in sorted(BASE_PATH.glob("*.md")):
        text = fp.read_text(encoding="utf-8", errors="ignore")
        low = text.lower()

        score = 0
        best_idx = None
        best_tok = None

        for tok in q_tokens:
            c = low.count(tok)
            if c > 0:
                score += c
                if best_idx is None:
                    best_idx = low.find(tok)
                    best_tok = tok

        if score <= 0:
            continue

        idx = best_idx if best_idx is not None else 0
        start = max(0, idx - 250)
        end = min(len(text), idx + 250)
        snippet = text[start:end].replace("\n", " ")

        hits.append((score, fp.name, best_tok, snippet))

    hits.sort(reverse=True, key=lambda x: x[0])
    hits = hits[:k]

    if not hits:
        return "No matches."

    out = []
    out.append(f"Tokens usados: {q_tokens}")
    for score, fname, tok, snippet in hits:
        out.append(f"- file: {fname} | score: {score} | first_hit: {tok}\n  snippet: ...{snippet}...")
    return "\n".join(out)

@tool
def rag_answer(question: str) -> str:
    """
    Retrieval + Generation:
    1) recupera evidencia con rag_search
    2) pide al LLM responder SOLO con esa evidencia + 'Fuentes'
    """
    evidence = rag_search.invoke({"query": question, "k": 4})
    ev = str(evidence)

    if "No matches." in ev:
        return "No tengo evidencia en los documentos locales para responder."

    prompt = f"""
Responde la pregunta usando SOLO la evidencia proporcionada.
Si falta información, dilo.
Al final añade una línea: Fuentes: <lista de archivos mencionados>

EVIDENCIA:
{ev}

PREGUNTA:
{question}
"""
    ans = llm.invoke(prompt)
    return ans.content

# Quiz y comparación


*   make_quiz: Kahoot rápido basado en evidencia


*   compare_animals: compara dos animales con fuentes








In [ ]:
@tool
def make_quiz(animal: str, n: int = 5, lang: str = "es") -> str:
    """
    Genera preguntas tipo test basadas SOLO en el doc local del animal.
    """
    ensure = ensure_animal.invoke({"animal": animal, "lang": lang})
    fp = _find_doc_for(animal, lang=lang)
    if fp is None:
        return f"[ERROR] no se encontró doc tras ensure: {ensure}"

    text = fp.read_text(encoding="utf-8", errors="ignore")

    prompt = f"""
Crea {n} preguntas tipo test (A/B/C/D) basadas SOLO en este documento.
Incluye la respuesta correcta y una justificación breve citando una frase del doc.

DOCUMENTO ({fp.name}):
{text[:8000]}
"""
    ans = llm.invoke(prompt)
    return ans.content

@tool
def compare_animals(a: str, b: str, lang: str = "es") -> str:
    """
    Compara dos animales usando SOLO evidencia recuperada del RAG local.
    """
    ensure_animal.invoke({"animal": a, "lang": lang})
    ensure_animal.invoke({"animal": b, "lang": lang})

    # Recuperamos evidencia para ambos
    ev_a = rag_search.invoke({"query": a, "k": 3})
    ev_b = rag_search.invoke({"query": b, "k": 3})

    prompt = f"""
Compara {a} vs {b} usando SOLO la evidencia.
Da una tabla con 3 filas: "Qué es", "Hábitat/entorno", "Dato distintivo".
Si algo no está, pon "No consta en la evidencia".
Al final: Fuentes: ...

EVIDENCIA {a}:
{ev_a}

EVIDENCIA {b}:
{ev_b}
"""
    ans = llm.invoke(prompt)
    return ans.content

# Crear el agente con AgentExecutor

El executor se encarga del bucle de tool-calling.

ASK mucho más corto que en Agent1.

In [ ]:
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import MessagesPlaceholder


tools = [
    list_docs,
    fetch_wikipedia,
    ensure_animal,
    add_note,
    rag_search,
    rag_answer,
    make_quiz,
    compare_animals,
]

system = f"""
Eres un agente que mantiene una mini base de conocimiento local en {BASE_PATH}.
Reglas:
- Si el usuario pregunta por un animal y no existe en docs, usa ensure_animal.
- Para responder preguntas de conocimiento, usa rag_answer (y por tanto evidencia).
- Si el usuario pide ver evidencias, usa rag_search.
- Para enriquecer la base, usa add_note.
- Para actividades docentes, usa make_quiz.
- No inventes información fuera de la evidencia local.
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

tool_agent = create_tool_calling_agent(llm, tools, prompt)

agent_executor = AgentExecutor(
    agent=tool_agent,
    tools=tools,
    verbose=True,
    max_iterations=8
)

def ask(q: str) -> str:
    return agent_executor.invoke({"input": q})["output"]

NameError: name 'list_docs' is not defined

# Prueba mínima

Antes de la demo completa, comprobamos que el agente puede hacer algo simple: listar docs.

In [ ]:
print(ask("Lista los documentos disponibles en la carpeta local."))

NameError: name 'ask' is not defined

# DEMO GUIADA

Demo 1 — Ingesta: crear/asegurar animal desde Wikipedia

El agente debe:

* comprobar si existe el doc local

* si no existe, llamar a Wikipedia

* guardar el .md en /content/docs

* decirte qué archivo ha creado

In [ ]:
print(ask("Asegura que exista un documento local del elefante en español. Si no existe, créalo desde Wikipedia. Devuélveme el nombre del archivo."))



> Entering new AgentExecutor chain...

Invoking: `ensure_animal` with `{'animal': 'elefante', 'lang': 'es'}`


OK: guardado animal_elephantidae_es.md

/tmp/ipython-input-410/780636563.py:64: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  md.append(f"- fetched_at: {datetime.utcnow().isoformat()}Z")


El documento local del elefante en español se ha creado/guardado como **animal_elephantidae_es.md**.

> Finished chain.
El documento local del elefante en español se ha creado/guardado como **animal_elephantidae_es.md**.


# Verificar que el documento está en la base local

Listamos los docs disponibles y abrimos el del elefante para enseñar “memoria en disco”.

In [ ]:
print(ask("Lista los documentos disponibles en la carpeta local."))



> Entering new AgentExecutor chain...

Invoking: `list_docs` with `{}`


['/content/docs/animal_elephantidae_es.md']Hay 1 documento disponible en la carpeta local:

- `/content/docs/animal_elephantidae_es.md`

> Finished chain.
Hay 1 documento disponible en la carpeta local:

- `/content/docs/animal_elephantidae_es.md`


# Mostrar el contenido del doc (sin LLM)

In [ ]:
from pathlib import Path

candidates = sorted(Path("/content/docs").glob("animal_*_es.md"))
print("Docs:", [c.name for c in candidates])

latest = max(candidates, key=lambda p: p.stat().st_mtime)
print("\nMostrando:", latest.name)
print(latest.read_text(encoding="utf-8")[:1200])

Docs: ['animal_elephantidae_es.md']

Mostrando: animal_elephantidae_es.md
# Elephantidae

Fuente: https://es.wikipedia.org/wiki/Elephantidae

## Resumen (Wikipedia)
Los elefantes o elefántidos (Elephantidae) son una familia de mamíferos placentarios del orden proboscideos. Antiguamente se clasificaban, junto con otros mamíferos de piel gruesa, en el orden, ahora inválido, de los paquidermos (Pachydermata). Existen hoy en día tres especies y diversas subespecies. Entre los géneros extintos de esta familia destacan los mamuts.

## Metadata
- lang: es
- fetched_at: 2026-03-02T17:01:49.359681Z



**RAG (retrieval visible): rag_search**

NO queremos respuesta final. Solo queremos ver qué evidencia recupera el sistema del documento local.

In [ ]:
print(ask("Usa rag_search para recuperar evidencias sobre: 'cuántas especies existen hoy en día'. Devuélveme solo los snippets y archivos."))



> Entering new AgentExecutor chain...


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '50', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1772496000000'}, 'provider_name': None}}, 'user_id': 'user_3AORm5UXrlqX9smEN1DY17ZMvez'}

**RAG (respuesta final): rag_answer**

El agente debe responder usando SOLO la evidencia local y terminar con Fuentes:.

In [ ]:
print(ask("Según los documentos locales, ¿cuántas especies existen hoy en día en Elephantidae? Responde en una frase y termina con Fuentes."))



> Entering new AgentExecutor chain...

Invoking: `rag_answer` with `{'question': '¿Cuántas especies existen hoy en día en Elephantidae?'}`


No tengo evidencia en los documentos locales para responder.

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '50', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1772496000000'}, 'provider_name': None}}, 'user_id': 'user_3AORm5UXrlqX9smEN1DY17ZMvez'}

In [ ]:
!pip -q install -U langchain langchain-core langchain-community langchain-openai langchain-groq

In [ ]:
import os, getpass

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("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")
print("OK: GROQ_API_KEY cargada")

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··········
OK: GROQ_API_KEY cargada


In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
    max_retries=2,
    timeout=30,
)

# test rápido
print(llm.invoke("Responde en una frase: ¿qué es un agente en IA?").content)

Un agente en inteligencia artificial (IA) es un programa o sistema que interactúa con un entorno y toma decisiones para lograr objetivos específicos, utilizando la información disponible y ajustando su comportamiento en función de las consecuencias de sus acciones.


In [ ]:
print(ask("Asegura que exista un documento local del elefante en español. Si no existe, créalo desde Wikipedia. Devuélveme el nombre del archivo."))